# 01 · Data overview and cleaning

This notebook checks the structure and quality of the bookings data, documents every cleaning decision and saves a clean file used by the analysis, the model and the Tableau dashboard.

**Dataset:** Hotel Booking Demand (Antonio, Almeida & Nunes, 2019). Bookings of a resort hotel and a city hotel in Portugal, arrivals from July 2015 to August 2017. The data is anonymised: there is no booking ID or guest name.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/hotel_bookings.csv")
print(df.shape)
df.head()

## 1.1 Structure

In [ ]:
df.info()

119,390 bookings and 32 columns. Types to fix or handle:
- `agent` and `company` are IDs stored as floats only because they contain NaN: they are categories, not numbers.
- `children` is a float for the same reason.
- `reservation_status_date` is text and must be converted to a date.
- The arrival date is split into year, month name and day: a real date column is built in 1.6.

## 1.2 Missing values

In [ ]:
df.isna().sum().loc[lambda s: s > 0].sort_values(ascending=False)

| Column | Missing | Meaning | Treatment |
|---|---|---|---|
| `company` | 112,593 (94.3%) | Not a corporate booking | Replaced by a `has_company` flag |
| `agent` | 16,340 (13.7%) | Booked without an agent | Filled with 0 = no agent |
| `country` | 488 (0.4%) | Truly missing | Filled with "Unknown" |
| `children` | 4 | Truly missing | Filled with 0 |

`agent` and `company` are not errors: an empty value carries meaning.

## 1.3 Duplicates

Without a booking ID, two identical rows can be an error **or** two identical rooms booked together (e.g. a group). Before deciding, the impact on the main metric is measured.

In [ ]:
n_dup = df.duplicated().sum()
print(f"Duplicate rows: {n_dup} ({n_dup / len(df):.1%})")
print(f"Rows in groups of identical rows (keep=False): {df.duplicated(keep=False).sum()}")

df_dedup = df.drop_duplicates()
print(f"Cancellation rate, all rows:       {df['is_canceled'].mean():.1%}")
print(f"Cancellation rate, no duplicates:  {df_dedup['is_canceled'].mean():.1%}")

In [ ]:
dup_rows = df[df.duplicated(keep=False)]
unique_rows = df[~df.duplicated(keep=False)]

print(f"Cancellation rate, duplicated rows: {dup_rows['is_canceled'].mean():.1%}")
print(f"Cancellation rate, unique rows:     {unique_rows['is_canceled'].mean():.1%}")
print()
print(dup_rows["deposit_type"].value_counts(normalize=True).round(3))
print()
print(dup_rows["market_segment"].value_counts(normalize=True).round(3).head())

- 31,994 rows (26.8%) are exact copies of another row; in total 40,165 rows belong to 8,171 groups of identical rows.
- Dropping them would cut the cancellation rate from **37.0% to 27.5%**: the duplicates are not random.
- Duplicated rows are cancelled far more often (**58.4% vs 26.2%**) and are mostly **Groups and Offline TA/TO (72%)**, with a large share of "Non Refund" deposits.

This matches how hotels work with tour operators: blocks of identical rooms are booked months ahead and released if they are not sold. Once anonymised, each room of a block becomes an identical row.

**Decision:** all rows are kept (dropping them would remove the largest cancellations) and flagged with `is_duplicate`, so that results can be checked without them.

In [ ]:
df["is_duplicate"] = df.duplicated(keep=False).astype(int)
df["is_duplicate"].sum()

## 1.4 Impossible and unusual values

In [ ]:
no_guests = (df["adults"] + df["children"].fillna(0) + df["babies"]) == 0
print(no_guests.sum(), "bookings with no guests")
df.loc[no_guests, ["hotel", "is_canceled", "market_segment", "adr", "stays_in_week_nights"]].head()

In [ ]:
print(df["adr"].describe())
print()
print((df["adr"] < 0).sum(), "rows with negative adr")
print((df["adr"] == 0).sum(), "rows with adr = 0")
df.sort_values("adr", ascending=False)[["hotel", "adr", "adults", "stays_in_week_nights", "is_canceled"]].head(3)

In [ ]:
df.loc[df["adr"] == 0, "market_segment"].value_counts()

| Case | Rows | Decision |
|---|---|---|
| No adults, children or babies | 180 | **Removed**: a stay with no guests is not possible (mostly Groups/Corporate with `adr` = 0, likely held rooms or entry errors) |
| `adr` = 5,400 (1 night, city hotel) | 1 | **Removed**: 10x the next highest value (510), almost certainly a typo |
| Negative `adr` (−6.38) | 1 | **Removed**: impossible price |
| `adr` = 0 | 1,959 | **Kept** for counts and cancellation rates, **excluded from ADR**: 680 are Complementary (free stays by definition), the rest may be package deals or entry errors |

## 1.5 Apply the cleaning decisions

In [ ]:
df_clean = df.copy()

# Impossible rows
no_guests = (df_clean["adults"] + df_clean["children"].fillna(0) + df_clean["babies"]) == 0
bad_adr = (df_clean["adr"] < 0) | (df_clean["adr"] > 5000)
df_clean = df_clean[~no_guests & ~bad_adr]

# Missing values
df_clean["children"] = df_clean["children"].fillna(0).astype(int)
df_clean["country"] = df_clean["country"].fillna("Unknown")
df_clean["agent"] = df_clean["agent"].fillna(0).astype(int)        # 0 = no agent
df_clean["has_company"] = df_clean["company"].notna().astype(int)  # 1 = corporate booking
df_clean = df_clean.drop(columns="company")

# Types
df_clean["reservation_status_date"] = pd.to_datetime(df_clean["reservation_status_date"])

print(len(df), "->", len(df_clean), "rows")
print(df_clean.isna().sum().sum(), "missing values left")

## 1.6 New columns

In [ ]:
df_clean["arrival_date"] = pd.to_datetime(
    df_clean["arrival_date_year"].astype(str) + "-"
    + df_clean["arrival_date_month"] + "-"
    + df_clean["arrival_date_day_of_month"].astype(str),
    format="%Y-%B-%d",
)
df_clean["total_nights"] = df_clean["stays_in_weekend_nights"] + df_clean["stays_in_week_nights"]
df_clean["revenue"] = df_clean["adr"] * df_clean["total_nights"]

print(df_clean["arrival_date"].min(), "->", df_clean["arrival_date"].max())
print((df_clean["total_nights"] == 0).sum(), "bookings with 0 nights")
df_clean["total_nights"].describe()

- Arrivals from 1 July 2015 to 31 August 2017: only 2016 is a full year, so months are compared, not yearly totals.
- Median stay of 3 nights. 645 bookings (0.5%) have 0 nights (likely day use): kept, with revenue 0. The longest stay (69 nights) is rare but plausible.

## 1.7 Columns that are only known after booking

In [ ]:
pd.crosstab(df_clean["reservation_status"], df_clean["is_canceled"])

`reservation_status` maps one-to-one to `is_canceled` (Canceled and No-Show = 1, Check-Out = 0): it is written at the end of the booking's life. Using it in a model would be **data leakage**. The same applies to `reservation_status_date` and `assigned_room_type` (assigned at check-in). These columns are kept for reporting but excluded from the model. The 1,206 no-shows are counted as cancellations.

## 1.8 Save

In [ ]:
df_clean.to_csv("../data/processed/hotel_bookings_clean.csv", index=False)
print(df_clean.shape)

## Summary of cleaning decisions

| # | Finding | Decision |
|---|---|---|
| 1 | `children` (4) and `country` (488) missing | Filled with 0 and "Unknown" |
| 2 | Empty `agent`/`company` = no agent/company | `agent` = 0, `company` → `has_company` |
| 3 | 26.8% duplicated rows, mostly group blocks, cancelled 58.4% of the time | Kept and flagged with `is_duplicate` |
| 4 | 180 bookings with no guests | Removed |
| 5 | `adr` negative (1) and 5,400 (1) | Removed |
| 6 | 1,959 bookings with `adr` = 0 | Kept, excluded from ADR |
| 7 | 645 stays with 0 nights | Kept |
| 8 | `reservation_status`, `reservation_status_date`, `assigned_room_type` | Excluded from the model (leakage) |

**Result:** 119,208 clean bookings and 36 columns.